# NB_hospital_daily_summary
**Gold Table:** `caresync_databricks.gold.hospital_daily_summary`  
**Grain:** One row per hospital per calendar day  
**Driving Table:** `caresync_databricks.silver.lab_results` (grouped by `hospital_id`, `test_date`)  
**Load Strategy:** Full rebuild — `CREATE OR REPLACE TABLE ... AS SELECT`  
**Joins:**
- LEFT JOIN `hospitals` on `hospital_id`
- LEFT JOIN `calendar` on `date_key = date_format(test_date, 'yyyyMMdd')`

**Window Function:** `LAG()` partitioned by `hospital_id`, ordered by `summary_date` — day-over-day trend  
**Source:** STTM — Silver to Gold (CareSync Health Network) v1.0

In [0]:
dbutils.widgets.text("catalog", "caresync_databricks", "Catalog Name")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

In [0]:
%sql
CREATE OR REPLACE TABLE caresync_databricks.gold.hospital_daily_summary AS

WITH daily_agg AS (
    SELECT
        lr.hospital_id,
        lr.test_date                                                        AS summary_date,
        COUNT(*)                                                            AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                       AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                        AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
    FROM caresync_databricks.silver.lab_results lr
    GROUP BY lr.hospital_id, lr.test_date
),

with_window AS (
    SELECT
        da.hospital_id,
        h.hospital_name,
        da.summary_date,
        c.day_name,
        c.is_weekend,
        da.total_tests,
        da.unique_patients_tested,
        da.active_doctor_count,
        da.abnormal_test_count,
        da.critical_test_count,
        CAST(da.abnormal_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))   AS abnormal_rate_pct,
        CAST(da.critical_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))   AS critical_rate_pct,
        LAG(da.total_tests) OVER (
            PARTITION BY da.hospital_id ORDER BY da.summary_date
        )                                                                       AS prev_day_total_tests
    FROM daily_agg da
    LEFT JOIN caresync_databricks.silver.hospitals h
        ON da.hospital_id = h.hospital_id
    LEFT JOIN caresync_databricks.silver.calendar c
        ON c.date_key = CAST(DATE_FORMAT(da.summary_date, 'yyyyMMdd') AS INT)
)

SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    prev_day_total_tests,
    CAST(
        CASE
            WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
            ELSE (total_tests - prev_day_total_tests) * 100.0 / prev_day_total_tests
        END AS DECIMAL(6,2)
    )                                                                           AS test_volume_change_pct
FROM with_window

In [0]:
print("[STEP 4] Computing processed record count")
record_count = spark.table(f"caresync_databricks.gold.hospital_daily_summary").count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))